# 0418_2 · Efecto sábado en validación con la base v4

**Por qué.** Con la regla del SLA adoptada, el sábado cuenta como día hábil y las guías que salen a zona ese día tienen una tasa de incumplimiento mucho mayor. Aquí se mide, con la base v4 y la validación de abril a agosto, cuánto de ese efecto se explica por el supuesto del calendario.

**Qué NO hace.** No toca la reserva de septiembre y octubre, no reentrena modelos y no cambia la etiqueta del modelado. Usa las predicciones de validación del modelo de clasificación de la decisión v2 (`pred_val_{modelo}.parquet`), la regla de capacidad fija (k alertas por día) y, para comparar, la tabla agregada de la prueba que dejó `0420_Interpretabilidad_5` (`05_alertas_por_segmento.csv`).

| Sección | Pregunta | Salida |
| --- | --- | --- |
| A | Incumplimiento, exhaustividad y precisión por día de la semana con k alertas diarias, en validación y en la prueba | `P9_a_*` |
| B | Plazo y margen en T0 según el día de la semana | `P9_b_*` |
| C | Etiqueta con el sábado como día no hábil, en las mismas guías de validación | `P9_c_*` |

La sección D de la versión anterior (umbral por día) ya no aplica: con capacidad fija cada día recibe las mismas k alertas.

**Regla de lectura de C (la misma de la v3, fijada antes de correr).** Si con el sábado no hábil el incumplimiento de las guías que salen el sábado queda **a lo sumo 2 puntos por encima** del promedio de lunes a viernes, el efecto sábado se atribuye al supuesto del calendario. Si sigue más de 2 puntos por encima, hay algo propio del sábado.

**Salidas.** CSV en `REVISION_2026_11/V4/SABADO/`. Solo agregados: ningún número de guía, cliente ni persona.

In [0]:
%pip install "duckdb<2" lightgbm xgboost
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 169.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 151.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [lightgbm]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, numpy as np, pandas as pd, duckdb
from MODULOS.modelado_temporal import alertas_por_capacidad
from sklearn.metrics import average_precision_score

PATH_DATABRICKS  = os.environ.get("PATH_DATABRICKS_0418", "/Volumes/proyectogrado/source/source/")
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
RUTA_GOLD_V3     = f"{CARPETA_SALIDAS}GOLD_V4_ENVIO.parquet"   # 0330 v4 (se conserva el nombre de la variable)
RUTA_CALENDARIO  = f"{PATH_DATABRICKS}PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
RUTA_PRUEBA_SEG  = f"{CARPETA_SALIDAS}INTERPRETACION/05_alertas_por_segmento.csv"   # salida agregada de 0420_5 (prueba sep–oct)
CARPETA_SAB      = f"{CARPETA_SALIDAS}SABADO/"
os.makedirs(CARPETA_SAB, exist_ok=True)

COL_FECHA_CAL   = "DATE"
HABIL_PRINCIPAL = "UPPER(TRIM(TIPO_DIA)) = 'DIA HABIL' AND FECHA_FESTIVA IS NULL"          # igual que 0330 y 0340
SIN_SABADO      = HABIL_PRINCIPAL + f' AND dayofweek(CAST("{COL_FECHA_CAL}" AS DATE)) <> 6'  # DuckDB: 6 = sábado
CORTE_OBSERVACION = json.load(open(f"{CARPETA_SALIDAS}P3_manifiesto.json"))["parametros"]["CORTE_OBSERVACION"]   # corte real de la v4
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]   # pandas: 0 = lunes

try:
    display
except NameError:
    display = print

def guardar(nombre, df, index=False):
    df.to_csv(f"{CARPETA_SAB}P9_{nombre}.csv", index=index)
    display(df)
    return df

In [0]:
# Decisión v2 registrada en 0416_2 (solo se lee)
DEC = json.load(open(f"{CARPETA_SALIDAS}P4b_decision_v2.json"))
MODELO_CLF, K = DEC["modelo_clf"], int(DEC["regla_alertas"]["k_por_dia"])
print(f"Modelo de clasificación: {MODELO_CLF} · capacidad: {K} alertas por día · corte: {CORTE_OBSERVACION}")

# Predicciones de validación del modelo elegido (versión 'internas'), abril a agosto
pv = pd.read_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO_CLF}.parquet", filters=[("version", "==", "internas"), ("tarea", "==", "clf")])
pv = pv[pv.modelo == MODELO_CLF].copy()
pv["dia"] = pd.to_datetime(pv["dia"])
pv["dia_semana"] = pv.dia.dt.dayofweek
pv["alerta"] = alertas_por_capacidad(pv.pred, pv.dia, K)   # las K de mayor riesgo de cada día
print(f"{len(pv):,} guías de validación · {pv.dia.nunique()} días · bloques {sorted(pv.bloque.astype(str).unique())}")

Modelo de clasificación: RandomForest · capacidad: 500 alertas por día · corte: 2026-08-28 18:25:20
966,041 guías de validación · 118 días · bloques ['2026-04-01', '2026-05-01', '2026-06-01', '2026-07-01', '2026-08-01']


## A. Incumplimiento y alertas por día de la semana en validación

`alertas_por_dia` es el total de alertas de ese día de la semana dividido entre **todos** los días del periodo (la columna suma K); `alertas_por_dia_propio` divide entre los días de ese tipo (con capacidad fija es K, salvo días con menos guías que K). Lo que interesa es la tasa real, la precisión y la exhaustividad de cada día.

In [0]:
def por_dia_semana(c, periodo):
    n_dias = c.dia.nunique()
    filas = []
    for ds, g in c.groupby("dia_semana"):
        filas.append({"periodo": periodo, "dia_semana": DIAS[int(ds)], "guias": len(g), "dias_de_ese_tipo": g.dia.nunique(),
                      "tasa_real": g.y.mean(), "alertas": int(g.alerta.sum()),
                      "alertas_por_dia": g.alerta.sum() / n_dias,
                      "alertas_por_dia_propio": g.alerta.sum() / g.dia.nunique(),
                      "precision": g.loc[g.alerta, "y"].mean() if g.alerta.any() else np.nan,
                      "exhaustividad": g.loc[g.y == 1, "alerta"].mean() if (g.y == 1).any() else np.nan})
    t = pd.DataFrame(filas)
    t["pct_de_las_alertas"] = 100 * t.alertas / t.alertas.sum()
    t["pct_de_las_guias"] = 100 * t.guias / t.guias.sum()
    return t

a_val = por_dia_semana(pv, "validación abr–ago")
a_bloques = pd.concat([por_dia_semana(g, f"validación {b}") for b, g in pv.groupby(pv.bloque.astype(str))], ignore_index=True)
guardar("a_validacion_dia_semana", pd.concat([a_val, a_bloques], ignore_index=True).round(4))

periodo,dia_semana,guias,dias_de_ese_tipo,tasa_real,alertas,alertas_por_dia,alertas_por_dia_propio,precision,exhaustividad,pct_de_las_alertas,pct_de_las_guias
validación abr–ago,Lunes,74619,17,0.06,7502,63.5763,441.2941,0.1954,0.3276,12.934,7.7242
validación abr–ago,Martes,174565,20,0.0658,10000,84.7458,500.0,0.3018,0.2629,17.2408,18.0701
validación abr–ago,Miércoles,210202,21,0.0692,10500,88.9831,500.0,0.3186,0.2301,18.1028,21.7591
validación abr–ago,Jueves,190941,20,0.067,10000,84.7458,500.0,0.327,0.2556,17.2408,19.7653
validación abr–ago,Viernes,172341,18,0.0672,9000,76.2712,500.0,0.2877,0.2234,15.5167,17.8399
validación abr–ago,Sábado,142149,21,0.1337,10500,88.9831,500.0,0.5802,0.3206,18.1028,14.7146
validación abr–ago,Domingo,1224,1,0.0049,500,4.2373,500.0,0.002,0.1667,0.862,0.1267
validación 2026-04-01,Lunes,18641,4,0.0619,2000,83.3333,500.0,0.202,0.3501,16.6667,9.6479
validación 2026-04-01,Martes,44179,4,0.0628,2000,83.3333,500.0,0.4545,0.3277,16.6667,22.8653
validación 2026-04-01,Miércoles,43419,5,0.0702,2500,104.1667,500.0,0.3356,0.2754,20.8333,22.472


,periodo,dia_semana,guias,dias_de_ese_tipo,tasa_real,alertas,alertas_por_dia,alertas_por_dia_propio,precision,exhaustividad,pct_de_las_alertas,pct_de_las_guias
0,validación abr–ago,Lunes,74619,17,0.0600,7502,63.5763,441.2941,0.1954,0.3276,12.9340,7.7242
1,validación abr–ago,Martes,174565,20,0.0658,10000,84.7458,500.0000,0.3018,0.2629,17.2408,18.0701
2,validación abr–ago,Miércoles,210202,21,0.0692,10500,88.9831,500.0000,0.3186,0.2301,18.1028,21.7591
3,validación abr–ago,Jueves,190941,20,0.0670,10000,84.7458,500.0000,0.3270,0.2556,17.2408,19.7653
4,validación abr–ago,Viernes,172341,18,0.0672,9000,76.2712,500.0000,0.2877,0.2234,15.5167,17.8399
5,validación abr–ago,Sábado,142149,21,0.1337,10500,88.9831,500.0000,0.5802,0.3206,18.1028,14.7146
6,validación abr–ago,Domingo,1224,1,0.0049,500,4.2373,500.0000,0.0020,0.1667,0.8620,0.1267
7,validación 2026-04-01,Lunes,18641,4,0.0619,2000,83.3333,500.0000,0.2020,0.3501,16.6667,9.6479
8,validación 2026-04-01,Martes,44179,4,0.0628,2000,83.3333,500.0000,0.4545,0.3277,16.6667,22.8653
9,validación 2026-04-01,Miércoles,43419,5,0.0702,2500,104.1667,500.0000,0.3356,0.2754,20.8333,22.4720


In [0]:
# Comparación con la prueba sep–oct (tabla agregada que ya dejó 0420_5; no se recalcula nada de la prueba)
if os.path.exists(RUTA_PRUEBA_SEG):
    pr = pd.read_csv(RUTA_PRUEBA_SEG)
    pr = pr[pr.segmento == "dia_semana"].copy()
    pr["dia_semana"] = pr.valor.astype(int).map(dict(enumerate(DIAS)))
    pr["pct_de_las_alertas"] = 100 * pr.alertas_por_dia / pr.alertas_por_dia.sum()
    comp = a_val[["dia_semana", "tasa_real", "alertas_por_dia", "pct_de_las_alertas", "precision", "exhaustividad"]].merge(
        pr[["dia_semana", "tasa_real", "alertas_por_dia", "pct_de_las_alertas", "precision_alertas", "exhaustividad"]],
        on="dia_semana", how="outer", suffixes=("_validacion", "_prueba"))
    comp = comp.sort_values("dia_semana", key=lambda s: s.map({d: i for i, d in enumerate(DIAS)}))
    guardar("a_validacion_vs_prueba", comp.round(4))
else:
    print("No está 05_alertas_por_segmento.csv de 0420_5: se omite la comparación con la prueba.")

sab = a_val[a_val.dia_semana == "Sábado"].iloc[0]
lv = a_val[a_val.dia_semana.isin(DIAS[:5])]
print(f"Validación · sábado: incumplimiento {sab.tasa_real:.1%}, precisión {sab.precision:.2f}, exhaustividad {sab.exhaustividad:.2f} · "
      f"lunes a viernes: incumplimiento {np.average(lv.tasa_real, weights=lv.guias):.1%}, precisión {lv.precision.min():.2f}–{lv.precision.max():.2f}")

dia_semana,tasa_real_validacion,alertas_por_dia_validacion,pct_de_las_alertas_validacion,precision,exhaustividad_validacion,tasa_real_prueba,alertas_por_dia_prueba,pct_de_las_alertas_prueba,precision_alertas,exhaustividad_prueba
Lunes,0.06,63.5763,12.934,0.1954,0.3276,0.0474,74.0741,14.8148,0.1048,0.2353
Martes,0.0658,84.7458,17.2408,0.3018,0.2629,0.0543,83.3333,16.6667,0.1702,0.1759
Miércoles,0.0692,88.9831,18.1028,0.3186,0.2301,0.0775,83.3333,16.6667,0.2298,0.1578
Jueves,0.067,84.7458,17.2408,0.327,0.2556,0.0671,83.3333,16.6667,0.1813,0.1614
Viernes,0.0672,76.2712,15.5167,0.2877,0.2234,0.0674,83.3333,16.6667,0.1789,0.1594
Sábado,0.1337,88.9831,18.1028,0.5802,0.3206,0.1429,83.3333,16.6667,0.6133,0.3435
Domingo,0.0049,4.2373,0.862,0.002,0.1667,0.0097,9.2593,1.8519,0.01,1.0


Validación · sábado: incumplimiento 13.4%, precisión 0.58, exhaustividad 0.32 · lunes a viernes: incumplimiento 6.7%, precisión 0.20–0.33


## B. Plazo y margen en T0 según el día de la semana

Si el sábado cuenta como hábil, una guía admitida el viernes con plazo de un día vence el sábado a medianoche. Aquí se mira, en las guías de validación, el plazo en días, el margen en T0 y qué parte sale con poco margen.

In [0]:
G   = f"read_parquet('{RUTA_GOLD_V3}')"
CAL = f"read_parquet('{RUTA_CALENDARIO}')"
con = duckdb.connect()
con.register("val", pv[["guia", "bloque", "dia_semana", "y", "pred", "alerta"]].assign(bloque=pv.bloque.astype(str)))

b = con.execute(f"""
    SELECT V.dia_semana, COUNT(*) AS guias,
           AVG(G.sla_dias) AS plazo_medio_dias,
           AVG((G.sla_dias = 1)::INT) AS pct_plazo_1_dia,
           MEDIAN(G.margen_sla_en_t0_h) AS margen_mediano_h,
           AVG((G.margen_sla_en_t0_h < 12)::INT) AS pct_margen_menor_12h,
           AVG((dayofweek(CAST(G.vencimiento_sla AS DATE)) = 6)::INT) AS pct_vence_en_sabado
    FROM val V JOIN {G} G USING (guia)
    GROUP BY 1 ORDER BY 1""").df()
b["dia_semana"] = b.dia_semana.astype(int).map(dict(enumerate(DIAS)))
guardar("b_plazo_margen_dia_semana", b.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dia_semana,guias,plazo_medio_dias,pct_plazo_1_dia,margen_mediano_h,pct_margen_menor_12h,pct_vence_en_sabado
Lunes,74619,1.44,0.6534,15.7989,0.0019,9.0E-4
Martes,174565,1.199,0.8461,14.7819,0.0118,9.0E-4
Miércoles,210202,1.1692,0.861,14.5883,0.0102,0.0057
Jueves,190941,1.1953,0.8412,14.6417,0.0069,0.0141
Viernes,172341,1.1919,0.8499,14.6544,0.008,0.13
Sábado,142149,1.1583,0.8787,15.4792,0.0014,0.8301
Domingo,1224,1.0,1.0,39.8083,0.0,0.0


,dia_semana,guias,plazo_medio_dias,pct_plazo_1_dia,margen_mediano_h,pct_margen_menor_12h,pct_vence_en_sabado
0,Lunes,74619,1.4400,0.6534,15.7989,0.0019,0.0009
1,Martes,174565,1.1990,0.8461,14.7819,0.0118,0.0009
2,Miércoles,210202,1.1692,0.8610,14.5883,0.0102,0.0057
3,Jueves,190941,1.1953,0.8412,14.6417,0.0069,0.0141
4,Viernes,172341,1.1919,0.8499,14.6544,0.0080,0.1300
5,Sábado,142149,1.1583,0.8787,15.4792,0.0014,0.8301
6,Domingo,1224,1.0000,1.0000,39.8083,0.0000,0.0000


## C. Etiqueta con el sábado como día no hábil (mismas guías de validación)

Se recalcula el vencimiento con la misma regla de 0340 (fin del n-ésimo día hábil desde la admisión, sin contar el día de admisión), cambiando solo el calendario: el sábado deja de ser hábil. Primero se reproduce la regla principal como control; debe coincidir con la etiqueta del modelado en casi todas las guías.

Después se compara, por día de la semana de la salida, el incumplimiento con las dos reglas y cómo se ven las alertas del modelo congelado frente a la etiqueta alternativa. El modelo **no** se reentrena: la pregunta es cuánto del «exceso» del sábado desaparece solo con cambiar el calendario.

In [0]:
def limite_variante(condicion_habil):
    # Límite (inicio del día siguiente al vencimiento) por guía de validación con la regla de 0340, para un calendario dado.
    return con.execute(f"""
    WITH C AS (SELECT CAST("{COL_FECHA_CAL}" AS DATE) AS fecha, bool_and(COALESCE({condicion_habil}, FALSE)) AS es_habil
               FROM {CAL} GROUP BY 1),
    HAB AS (SELECT fecha, es_habil, SUM(es_habil::INT) OVER (ORDER BY fecha) AS n_hab FROM C),
    HABIL AS (SELECT fecha, n_hab FROM HAB WHERE es_habil),
    POB AS (SELECT V.guia, G.FECHA_ENVIO AS ini, G.fecha_entrega, G.t0, CAST(G.sla_dias AS INT) AS d
            FROM val V JOIN {G} G USING (guia))
    SELECT P.guia, P.t0, P.fecha_entrega, CAST(HV.fecha AS TIMESTAMP) + INTERVAL 1 DAY AS lim
    FROM POB P
    LEFT JOIN HAB H0 ON H0.fecha = CAST(P.ini AS DATE)
    LEFT JOIN HABIL HV ON HV.n_hab = H0.n_hab + GREATEST(P.d, 0)
                          + CASE WHEN P.d <= 0 AND NOT H0.es_habil THEN 1 ELSE 0 END""").df()

def etiqueta(t):
    lim, fe = pd.to_datetime(t.lim), pd.to_datetime(t.fecha_entrega)
    y = np.where(lim.isna(), np.nan,
                 np.where(fe.notna(), (fe >= lim).astype(float),
                          np.where(lim <= pd.Timestamp(CORTE_OBSERVACION), 1.0, np.nan)))
    return pd.Series(y, index=t.index)

def horas(a, b):
    return (pd.to_datetime(a) - pd.to_datetime(b)) / pd.Timedelta(hours=1)

base = limite_variante(HABIL_PRINCIPAL).rename(columns={"lim": "lim_principal"})
alt  = limite_variante(SIN_SABADO)[["guia", "lim"]].rename(columns={"lim": "lim_sin_sabado"})
c = pv[["guia", "dia", "dia_semana", "y", "pred", "alerta"]].merge(base, on="guia").merge(alt, on="guia")
c["y_principal"]   = etiqueta(c.rename(columns={"lim_principal": "lim"}))
c["y_sin_sabado"]  = etiqueta(c.rename(columns={"lim_sin_sabado": "lim"}))
c["margen_sin_sabado_h"] = horas(c.lim_sin_sabado, c.t0)

control = pd.DataFrame([{"guias_validacion": len(pv), "guias_unidas": len(c),
                         "pct_coincide_regla_principal": 100 * (c.y_principal == c.y).mean(),
                         "pct_sin_vencimiento_sin_sabado": 100 * c.lim_sin_sabado.isna().mean()}])
guardar("c_control", control.round(3))
assert control.pct_coincide_regla_principal.iloc[0] > 99, "La regla reproducida no coincide con la etiqueta del modelado: revisar antes de leer C"

guias_validacion,guias_unidas,pct_coincide_regla_principal,pct_sin_vencimiento_sin_sabado
966041,966041,100.0,0.0


In [0]:
cc = c.dropna(subset=["y_sin_sabado"])
filas = []
for ds, g in cc.groupby("dia_semana"):
    al = g.alerta
    filas.append({"dia_semana": DIAS[int(ds)], "guias": len(g),
                  "incumplimiento_principal": g.y.mean(), "incumplimiento_sin_sabado": g.y_sin_sabado.mean(),
                  "guias_que_dejan_de_incumplir": int(((g.y == 1) & (g.y_sin_sabado == 0)).sum()),
                  "margen_mediano_principal_h": horas(g.lim_principal, g.t0).median(),
                  "margen_mediano_sin_sabado_h": g.margen_sin_sabado_h.median(),
                  "alertas": int(al.sum()),
                  "precision_principal": g.loc[al, "y"].mean() if al.any() else np.nan,
                  "precision_sin_sabado": g.loc[al, "y_sin_sabado"].mean() if al.any() else np.nan,
                  "AP_principal": average_precision_score(g.y, g.pred) if g.y.nunique() == 2 else np.nan,
                  "AP_sin_sabado": average_precision_score(g.y_sin_sabado, g.pred) if g.y_sin_sabado.nunique() == 2 else np.nan})
c_tab = pd.DataFrame(filas)
guardar("c_etiqueta_sin_sabado_dia_semana", c_tab.round(4))

lab = c_tab[c_tab.dia_semana.isin(DIAS[:5])]
prom_lv = np.average(lab.incumplimiento_sin_sabado, weights=lab.guias)
sab = c_tab.loc[c_tab.dia_semana == "Sábado", "incumplimiento_sin_sabado"]
prom_lv_principal = np.average(lab.incumplimiento_principal, weights=lab.guias)
sab_principal = c_tab.loc[c_tab.dia_semana == "Sábado", "incumplimiento_principal"]
dif_principal = 100 * (float(sab_principal.iloc[0]) - prom_lv_principal) if len(sab_principal) else np.nan
dif = 100 * (float(sab.iloc[0]) - prom_lv) if len(sab) else np.nan
lectura_c = ("el efecto sábado se explica por el supuesto del calendario" if dif <= 2 else
             "queda un efecto propio del sábado")
resumen_c = pd.DataFrame([{"incumplimiento_global_principal": cc.y.mean(), "incumplimiento_global_sin_sabado": cc.y_sin_sabado.mean(),
                           "sabado_sin_sabado": float(sab.iloc[0]) if len(sab) else np.nan, "lunes_a_viernes_sin_sabado": prom_lv,
                           "diferencia_puntos_regla_principal": dif_principal,
                           "diferencia_puntos_sin_sabado": dif, "lectura": lectura_c}])
guardar("c_resumen", resumen_c.round(4))
print(f"Sábado frente a lunes–viernes: {dif_principal:+.1f} puntos con la regla principal; {dif:+.1f} con el sábado no hábil → {lectura_c}")

dia_semana,guias,incumplimiento_principal,incumplimiento_sin_sabado,guias_que_dejan_de_incumplir,margen_mediano_principal_h,margen_mediano_sin_sabado_h,alertas,precision_principal,precision_sin_sabado,AP_principal,AP_sin_sabado
Lunes,74619,0.06,0.0506,701,15.7989,16.7836,7502,0.1954,0.1622,0.2086,0.1656
Martes,174565,0.0658,0.0636,383,14.7819,14.8878,10000,0.3018,0.2873,0.2695,0.26
Miércoles,210202,0.0692,0.0689,58,14.5883,14.5992,10500,0.3186,0.3173,0.2732,0.2726
Jueves,190941,0.067,0.0668,40,14.6417,14.6453,10000,0.327,0.3265,0.2915,0.2917
Viernes,172341,0.0672,0.0647,431,14.6544,14.6544,9000,0.2877,0.282,0.2598,0.2594
Sábado,142149,0.1337,0.0284,14964,15.4792,63.6892,10500,0.5802,0.1263,0.4325,0.1192
Domingo,1224,0.0049,0.0049,0,39.8083,39.8083,500,0.002,0.002,0.0067,0.0067


incumplimiento_global_principal,incumplimiento_global_sin_sabado,sabado_sin_sabado,lunes_a_viernes_sin_sabado,diferencia_puntos_regla_principal,diferencia_puntos_sin_sabado,lectura
0.0765,0.0593,0.0284,0.0647,6.6986,-3.6323,el efecto sábado se explica por el supuesto del calendario


Sábado frente a lunes–viernes: +6.7 puntos con la regla principal; -3.6 con el sábado no hábil → el efecto sábado se explica por el supuesto del calendario


## Cómo leer y qué entregar

- **A** muestra cómo se comportan las K alertas diarias según el día de la semana en validación y en la prueba.
- **B y C** responden cuánto pesa el supuesto del sábado hábil. Esto **no cambia** los resultados del documento: la regla principal sigue siendo la de 0330 y la prueba no se toca. Sirve para la sección 5.6 y la discusión (6.3).

**Entregar:** la carpeta `V4/SABADO/` (solo CSV agregados).